In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
from ozone_extremes_leap import *
import matplotlib.pyplot as plt
import xarray as xr 
from mpl_toolkits.basemap import Basemap
from matplotlib.pyplot import cm
import seaborn as sns
from matplotlib import rc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def calc_parc_o3(var):
    
        m_air = 28.964/(6.022E23)
        g = 980.6
    
        if 'plev' in var.dims: 
            plev=var.plev
            level='plev'
        if 'lev' in var.dims: 
            plev=var.lev
            level='lev'
        if 'level' in var.dims: 
            plev=var.level
            level='level'
        
        
        time=var.time
        delta_p = np.zeros((len(time), len(plev)))
        
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] <= 1000: 
            factor=100
            factor_2=1 # conversion Pa->hPa
        if plev[0]>plev[len(plev)-1] and plev[0] <=1000: 
            factor=100
            factor_2=1
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] >1000: 
            factor=1
            factor_2=100
        if plev[0]>plev[len(plev)-1] and plev[0] >1000: 
            factor=1
            factor_2=100
        
        if plev[0]<plev[len(plev)-1]: # for pressure levels in hPa
            
            
            for levelx in range(1,len(plev)): delta_p[:,levelx].fill(plev[levelx] - plev[levelx-1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa
 
            O3 = var * weights_p * 10/ (g * m_air)
        
            if level=='plev': O3=O3.sel(plev=slice(30*factor_2,70*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(30*factor_2,70*factor_2))
            if level=='level': O3=O3.sel(level=slice(30*factor_2,70*factor_2))

            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
            
        if plev[0]>plev[len(plev)-1]: # for pressure levels in hPa
        
            
            for levelx in range(0,len(plev)-1): delta_p[:,levelx].fill(plev[levelx] - plev[levelx+1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa

            O3 = var * weights_p * 10/ (g * m_air)
            
            if level =='plev': O3=O3.sel(plev=slice(70*factor_2,30*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(70*factor_2,30*factor_2)) 
            if level=='level': O3=O3.sel(level=slice(70*factor_2,30*factor_2)) 
                
            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
    
        return O3.where(O3 != 0)

Import timeslice 2000 dataset:

In [ ]:
#WACCM INT_O3

nc_fid=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc')

In [ ]:
O3=nc_fid['O3']

O3=calc_parc_o3(O3)

O3=O3.mean(dim='lon')

var=O3.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var.lat))
var = var.weighted(weights)     
var=var.mean(dim='lat')

In [ ]:
#WACCM INT_O3 PSL

nc_fid_SURF=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.PSL.merged.nc')
PSL_clim=nc_fid_SURF['PSL']
PSL_clim=PSL_clim.groupby('time.dayofyear').mean()

Calculate ozone minimum years and plot ozone partial column in those years:

In [ ]:
%reload_ext autoreload
%autoreload 2


# add MERRA2 2020 event

fig, ax = plt.subplots(1,1,figsize=(13,10), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')

extreme_years=10
years = 39
model='WACCM'
lev='plev'


O3='O3'
#这里有xrray的不兼容问题
O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes(nc_fid, O3,  lev, years,extreme_years,model)
var_min_slp_x,xpt,ypt, var_anomalies_slp_x, significance_slp_x = analyse_ozone_extremes(nc_fid_SURF, 'PSL', extreme_years, True, O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years, model)

O3_neutral_years=np.zeros((years-extreme_years))

var_clim=var.groupby("time.dayofyear").mean("time")

x=0

O3_lowest=O3_years[O3_lowest]

for y in range(years):

    if np.isin(O3_years[y],O3_lowest)==False:

        O3_neutral_years[x]=O3_years[y]
        x=x+1

var_lowest = var.sel(time=var.time.dt.year.isin([O3_lowest]))
var_lowest_mean = np.array(var_lowest.groupby("time.dayofyear").mean("time"))
var_lowest_std = np.array(var_lowest.groupby("time.dayofyear").std("time"))

var_lowest_before = var.sel(time=var.time.dt.year.isin([O3_lowest-1]))
var_lowest_mean_before = np.array(var_lowest_before.groupby("time.dayofyear").mean("time"))
var_lowest_std_before = np.array(var_lowest_before.groupby("time.dayofyear").std("time"))

var_highest = var.sel(time=var.time.dt.year.isin([O3_highest]))
var_highest_mean = np.array(var_highest.groupby("time.dayofyear").mean("time"))
var_highest_std = np.array(var_highest.groupby("time.dayofyear").std("time"))

var_highest_before = var.sel(time=var.time.dt.year.isin([O3_highest-1]))
var_highest_mean_before = np.array(var_highest_before.groupby("time.dayofyear").mean("time"))
var_highest_std_before = np.array(var_highest_before.groupby("time.dayofyear").std("time"))


var_neutral = var.sel(time=var.time.dt.year.isin([O3_neutral_years]))
var_neutral_mean = np.array(var_neutral.groupby("time.dayofyear").mean("time"))
var_neutral_std = np.array(var_neutral.groupby("time.dayofyear").std("time"))


var_neutral_before = var.sel(time=var.time.dt.year.isin([O3_neutral_years-1]))
var_neutral_mean_before = np.array(var_neutral_before.groupby("time.dayofyear").mean("time"))
var_neutral_std_before = np.array(var_neutral_before.groupby("time.dayofyear").std("time"))

var_lowest=np.reshape(np.array(var_lowest), (extreme_years, 365))
var_lowest_before=np.reshape(np.array(var_lowest_before), (extreme_years, 365))

color = cm.twilight(np.linspace(0, 1, 10))

for j in range(extreme_years):
    ax.plot(range(91,365), var_lowest[j,0:365-91],  color=color[j], alpha=0.8,linewidth=2, label='low O3 years')
    ax.plot(range(0,91), var_lowest_before[j,365-92:365-1] , color=color[j], alpha=0.8,linewidth=2)

#  plt.errorbar(range(92,len(var_highest_mean)), var_highest_mean[0:len(var_lowest_mean)-92] ,var_highest_std[0:len(var_lowest_mean)-92], color='red', alpha=0.5,elinewidth=1.5, linewidth=3, label='INT_O3, high O3')
plot=ax.errorbar(range(91,365), var_neutral_mean[0:365-91] ,var_neutral_std[0:365-91], color='grey', alpha=0.5,elinewidth=1.5, linewidth=3, label='all other years')


# plt.errorbar(range(0,92), var_highest_mean_before[len(var_lowest_mean)-92:len(var_lowest_mean)] ,var_highest_std_before[len(var_lowest_mean)-92:len(var_lowest_mean)], color='red', alpha=0.5,elinewidth=1.5, linewidth=3)
ax.errorbar(range(0,91), var_neutral_mean_before[365-92:365-1] ,var_neutral_std[365-92:365-1], color='grey', alpha=0.5,elinewidth=1.5, linewidth=3)

ax.set_xlim(0,366)

ax.set_xticks([0,31,61,91,122,150,181,211,242,273,304,334])
ax.set_xticklabels( ['Oct', 'Nov', 'Dec', 'Jan','Feb','Mar','Apr','May','June','July','Aug','Sep'], fontsize=15)
#  ax[i].set_xticklabels( ['01.10.', '01.11.', '01.12.', '01.01.','01.02.','01.03.','01.04.','01.05.','01.06.','01.07.','01.08.','01.09.'], fontsize=15)

    
ax.set_ylabel('Partial ozone column, 30-70 hPa (DU)', fontsize=18)
ax.set_yticklabels(ax.get_yticks(), size = 18)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_10extr.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_10extr.png')

In [ ]:
m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l')
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

levels = np.linspace(-4,4, 16)
#这里也有不兼容问题
plot_slp=m.contourf(xpt,ypt,var_min_slp_x/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both')


#m.contourf(xpt, ypt,  significance_slp, colors='none', levels=[0,0.05, 100], hatches=['','....'])
plt.title('Mean surface response (n=10)')

In [ ]:
extreme_years=1

O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes(nc_fid, O3,  lev, years,extreme_years,model)
var_min_slp_008,xpt,ypt, var_anomalies_slp_x, significance_slp_x = analyse_ozone_extremes(nc_fid_SURF, 'PSL', extreme_years, True, O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years, model)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l')
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

levels = np.linspace(-4,4, 16)

plot_slp=m.contourf(xpt,ypt,var_min_slp_008/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both')
#m.contourf(xpt, ypt,  significance_slp, colors='none', levels=[0,0.05, 100], hatches=['','....'])
plt.title('surface response in year 008')

In [ ]:
#select year with the strongest ozone minima
var=var.sel(time=nc_fid.time.dt.month.isin([1,2,3,4,5,6]))
var_0008=var.sel(time=var.time.dt.year.isin([8]))
var_clim=var.groupby('time.dayofyear').mean()

## Ozone plots for different initialisations

In [ ]:
nc_fid_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F=nc_fid_F['O3']
o3_F=o3_F.mean(dim='lon')

o3_F=calc_parc_o3(o3_F)

var_F=o3_F.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_F.lat))
var_F = var_F.weighted(weights)     
var_F=var_F.mean(dim='lat')

In [ ]:
nc_fid_F2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F2=nc_fid_F2['O3']
o3_F2=o3_F2.mean(dim='lon')

o3_F2=calc_parc_o3(o3_F2)

var_F2=o3_F2.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_F2.lat))
var_F2 = var_F2.weighted(weights)     
var_F2=var_F2.mean(dim='lat')

In [ ]:
nc_fid_F3=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F3=nc_fid_F3['O3']
o3_F3=o3_F3.mean(dim='lon')

o3_F3=calc_parc_o3(o3_F3)

var_F3=o3_F3.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_F3.lat))
var_F3 = var_F3.weighted(weights)     
var_F3=var_F3.mean(dim='lat')

In [ ]:
nc_fid_FNC=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_FNC=nc_fid_FNC['O3']
o3_FNC=o3_FNC.mean(dim='lon')

o3_FNC=calc_parc_o3(o3_FNC)

var_FNC=o3_FNC.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_FNC.lat))
var_FNC = var_FNC.weighted(weights)     
var_FNC=var_FNC.mean(dim='lat')

In [ ]:
nc_fid_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M=nc_fid_M['O3']
o3_M=o3_M.mean(dim='lon')

o3_M=calc_parc_o3(o3_M)

var_M=o3_M.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_M.lat))
var_M = var_M.weighted(weights)     
var_M=var_M.mean(dim='lat')

In [ ]:
nc_fid_M2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M2=nc_fid_M2['O3']
o3_M2=o3_M2.mean(dim='lon')

o3_M2=calc_parc_o3(o3_M2)

var_M2=o3_M2.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_M2.lat))
var_M2 = var_M2.weighted(weights)     
var_M2=var_M2.mean(dim='lat')

In [ ]:
nc_fid_M3=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M3=nc_fid_M3['O3']
o3_M3=o3_M3.mean(dim='lon')

o3_M3=calc_parc_o3(o3_M3)

var_M3=o3_M3.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_M3.lat))
var_M3= var_M3.weighted(weights)     
var_M3=var_M3.mean(dim='lat')

In [ ]:
nc_fid_A2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc' ,concat_dim='member', combine='nested')

o3_A2=nc_fid_A2['O3']
o3_A2=o3_A2.mean(dim='lon')

o3_A2=calc_parc_o3(o3_A2)

var_A2=o3_A2.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_A2.lat))
var_A2 = var_A2.weighted(weights)     
var_A2=var_A2.mean(dim='lat')

In [ ]:
nc_fid_J=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc' ,concat_dim='member', combine='nested')

o3_J=nc_fid_J['O3']
o3_J=o3_J.mean(dim='lon')

o3_J=calc_parc_o3(o3_J)

var_J=o3_J.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_J.lat))
var_J = var_J.weighted(weights)     
var_J=var_J.mean(dim='lat')

In [ ]:
rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc('text', usetex=True)

fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")


ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)
ax.plot(range(len(var_clim)), var_clim , color='grey', linestyle=':', linewidth=3)

ax.plot(range(len(var_J.time)), var_J.mean(dim='member'), color='green',  linewidth=3, label='Jan')

for mem in range(len(var_J.member)):
    ax.plot(range(len(var_J.time)), var_J.sel(member=mem), color='green',  linewidth=0.8, alpha=0.5)


ax.plot(range(31,len(var_F.time)+31), var_F.mean(dim='member'), color='royalblue',  linewidth=3, label='Feb')

for mem in range(len(var_F.member)):
    ax.plot(range(31,len(var_F.time)+31), var_F.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)


ax.plot(range(59,len(var_M.time)+59), var_M.mean(dim='member'), color='hotpink',  linewidth=3, label='Mar')

for mem in range(len(var_M.member)):
    ax.plot(range(59,len(var_M.time)+59), var_M.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)


ax.set_xlim(0,150)
plt.legend(fontsize=22)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Partial ozone column, 30-70 hPa (DU)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008.png')

## SURFACE RESPONSE OF FEB INITIALISATION

In [ ]:
nc_fid_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_SURF/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

p_F=nc_fid_F['PSL']

In [ ]:
var_F_runmean=var_F.rolling(time=5,center=True).mean() #calculate 5-day running mean
var_F_runmean=var_F_runmean.sel(time=var_F_runmean.time.dt.month.isin([3,4])) #select March/April
var_F_runmean=var_F_runmean.groupby('member').argmin(dim='time') #find date (day after March 1st) of minimum ozone value in each member
print(np.array(var_F_runmean))

In [ ]:
p_F=p_F.groupby("time.dayofyear")-PSL_clim #remove daily climatology of timeslice simulation to get anomalies

p_F_time=p_F.sel(time=p_F.time.dt.month.isin([3,4,5])) 

p_F_time=p_F_time.groupby('member')

PSL=[]

i=0
for member,group in p_F_time:
    PSL.append(np.array(np.mean(group[int(var_F_runmean[i]):int(var_F_runmean[i])+30,:,:],axis=0))) #for each member, average over 30 days after ozone minima
    i=i+1

In [ ]:
PSL=np.nanmean(PSL, axis=0) #average over all members

In [ ]:
nc_fid_FNC=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL_SURF/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

p_FNC=nc_fid_FNC['PSL']

var_FNC_runmean=var_FNC.rolling(time=5,center=True).mean() #calculate 5-day running mean
var_FNC_runmean=var_FNC_runmean.sel(time=var_FNC_runmean.time.dt.month.isin([3,4])) #select March/April
var_FNC_runmean=var_FNC_runmean.groupby('member').argmin(dim='time') #find date (day after March 1st) of minimum ozone value in each member
print(np.array(var_FNC_runmean))
p_FNC=p_FNC.groupby("time.dayofyear")-PSL_clim #remove daily climatology of timeslice simulation to get anomalies

p_FNC_time=p_FNC.sel(time=p_FNC.time.dt.month.isin([3,4,5])) 

p_FNC_time=p_FNC_time.groupby('member')

PSL_NC=[]

i=0
for member,group in p_FNC_time:
    PSL_NC.append(np.array(np.mean(group[int(var_FNC_runmean[i]):int(var_FNC_runmean[i])+30,:,:],axis=0))) #for each member, average over 30 days after ozone minima
    i=i+1
    
PSL_NC=np.nanmean(PSL_NC, axis=0) #average over all members

print(PSL_NC)

In [ ]:
fig, ax = plt.subplots(1,4,figsize=(12,4), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[0])
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

levels = np.linspace(-6,6, 25)

plot_slp=m.contourf(xpt,ypt,var_min_slp_x/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[0])
ax[0].set_title('mean surface response \n after ozone minima \n (n=10)', fontsize=18)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[1])
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)


plot_slp=m.contourf(xpt,ypt,var_min_slp_008/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[1])
ax[1].set_title('surface response \n in year 008 \n (n=1)', fontsize=18)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[2])
lons_m,lats_m=np.meshgrid(p_F.lon,p_F.lat)
xpt2,ypt2 = m(lons_m,lats_m)
m.drawcoastlines(linewidth=0.3)

m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)


plot_slp=m.contourf(xpt2,ypt2,PSL/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[2])
ax[2].set_title('surface response, Feb, \n small pertlim \n (n=30)', fontsize=18)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[3])
lons_m,lats_m=np.meshgrid(p_F.lon,p_F.lat)
xpt2,ypt2 = m(lons_m,lats_m)
m.drawcoastlines(linewidth=0.3)

m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

plot_slp=m.contourf(xpt2,ypt2,PSL_NC/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[3])
ax[3].set_title('surface response, Feb, \n NOCOUPL \n (n=30)', fontsize=18)

cbar=fig.colorbar(plot_slp, ax=ax[0:4], location='bottom', shrink=0.5)
cbar.set_label(r"hPa",  fontsize=18)

plt.savefig('/home/weiji/test code/plots/slp_response_ozone_min_Feb.pdf', bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(31,len(var_F2.time)+31), var_F2.mean(dim='member'), color='royalblue',  linewidth=3, label='Feb, large pertlim')

for mem in range(len(var_F2.member)):
    ax.plot(range(31,len(var_F2.time)+31), var_F2.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)


ax.plot(range(59,len(var_M2.time)+59), var_M2.mean(dim='member'), color='hotpink',  linewidth=3, label='Mar, large pertlim')

for mem in range(len(var_M2.member)):
    ax.plot(range(59,len(var_M2.time)+59), var_M2.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)

ax.plot(range(90,len(var_A2.time)+90), var_A2.mean(dim='member'), color='orange',  linewidth=3, label='Apr, large pertlim')

for mem in range(len(var_A2.member)):
    ax.plot(range(90,len(var_A2.time)+90), var_A2.sel(member=mem), color='orange',  linewidth=0.8, alpha=0.5)

ax.set_xlim(0,150)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Partial ozone column, 30-70 hPa (DU)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.legend(fontsize=22)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008_large.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008_large.png')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(31,len(var_F.time)+31), var_F.mean(dim='member'), color='royalblue',  linewidth=3, label='Feb, small pertlim')

for mem in range(len(var_F.member)):
    ax.plot(range(31,len(var_F.time)+31), var_F.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_F2.time)+31), var_F2.mean(dim='member'), color='teal',  linewidth=3,label='Feb, large pertlim')

for mem in range(len(var_F2.member)):
    ax.plot(range(31,len(var_F2.time)+31), var_F2.sel(member=mem), color='teal',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_F3.time)+31), var_F3.mean(dim='member'), color='midnightblue',  linewidth=3, label='Feb, Q pert')

for mem in range(len(var_F3.member)):
    ax.plot(range(31,len(var_F3.time)+31), var_F3.sel(member=mem), color='midnightblue',  linewidth=0.8, alpha=0.5)

ax.set_xlim(0,150)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Partial ozone column, 30-70 hPa (DU)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.legend(fontsize=22)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008_Feb.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008_Feb.png')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(59,len(var_M.time)+59), var_M.mean(dim='member'), color='hotpink',  linewidth=3, label='Mar, small pertlim')

for mem in range(len(var_M.member)):
    ax.plot(range(59,len(var_M.time)+59), var_M.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)

ax.plot(range(59,len(var_M2.time)+59), var_M2.mean(dim='member'), color='purple',  linewidth=3, label='Mar, large pertlim')

for mem in range(len(var_M2.member)):
    ax.plot(range(59,len(var_M2.time)+59), var_M2.sel(member=mem), color='purple',  linewidth=0.8, alpha=0.5)

ax.plot(range(59,len(var_M3.time)+59), var_M3.mean(dim='member'), color='blueviolet',  linewidth=3, label='Mar, Q pert')

for mem in range(len(var_M3.member)):
    ax.plot(range(59,len(var_M3.time)+59), var_M3.sel(member=mem), color='blueviolet',  linewidth=0.8, alpha=0.5)

ax.set_xlim(0,150)

plt.legend(fontsize=22)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Partial ozone column, 30-70 hPa (DU)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008_Mar.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_0008_Mar.png')

## Wind plots for different intialisations

In [ ]:
#WACCM INT_O3

nc_fid=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc')

In [ ]:
O3=nc_fid['U']

O3=O3.sel(plev=1000)

O3=O3.sel(time=nc_fid.time.dt.month.isin([1,2,3,4,5,6]))
O3=O3.mean(dim='lon')

var=O3.sel(lat=60, method='nearest')

var_0008=var.sel(time=var.time.dt.year.isin([8]))

In [ ]:
var_0008_clim=var.groupby("time.dayofyear").mean()

In [ ]:
nc_fid_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F=nc_fid_F['U']
o3_F=o3_F.mean(dim='lon')

o3_F=o3_F.sel(plev=1000)

var_F=o3_F.sel(lat=60, method='nearest')

In [ ]:
nc_fid_F2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F2=nc_fid_F2['U']
o3_F2=o3_F2.mean(dim='lon')

o3_F2=o3_F2.sel(plev=1000)

var_F2=o3_F2.sel(lat=60, method='nearest')

In [ ]:
nc_fid_F3=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F3=nc_fid_F3['U']
o3_F3=o3_F3.mean(dim='lon')

o3_F3=o3_F3.sel(plev=1000)

var_F3=o3_F3.sel(lat=60, method='nearest')

In [ ]:
nc_fid_FNC=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_FNC=nc_fid_FNC['U']
o3_FNC=o3_FNC.mean(dim='lon')

o3_FNC=o3_FNC.sel(plev=1000)

var_FNC=o3_FNC.sel(lat=60, method='nearest')

In [ ]:
nc_fid_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M=nc_fid_M['U']
o3_M=o3_M.mean(dim='lon')

o3_M=o3_M.sel(plev=1000)

var_M=o3_M.sel(lat=60, method='nearest')

In [ ]:
nc_fid_M2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M2=nc_fid_M2['U']
o3_M2=o3_M2.mean(dim='lon')

o3_M2=o3_M2.sel(plev=1000)

var_M2=o3_M2.sel(lat=60, method='nearest')

In [ ]:
nc_fid_M3=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M3=nc_fid_M3['U']
o3_M3=o3_M3.mean(dim='lon')

o3_M3=o3_M3.sel(plev=1000)

var_M3=o3_M3.sel(lat=60, method='nearest')

In [ ]:
nc_fid_J=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc' ,concat_dim='member', combine='nested')

o3_J=nc_fid_J['U']
o3_J=o3_J.mean(dim='lon')

o3_J=o3_J.sel(plev=1000)

var_J=o3_J.sel(lat=60, method='nearest')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

# plt.errorbar(range(0,92), var_highest_mean_before[len(var_lowest_mean)-92:len(var_lowest_mean)] ,var_highest_std_before[len(var_lowest_mean)-92:len(var_lowest_mean)], color='red', alpha=0.5,elinewidth=1.5, linewidth=3)
ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)
ax.plot(range(len(var_0008_clim)), var_0008_clim , color='grey', linestyle=':', linewidth=3)


ax.plot(range(31,len(var_F.time)+31), var_F.mean(dim='member'), color='royalblue', label='Feb', linewidth=3)

for mem in range(len(var_F.member)):
    ax.plot(range(31,len(var_F.time)+31), var_F.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)


ax.plot(range(59,len(var_M.time)+59), var_M.mean(dim='member'), color='hotpink', label='Mar',  linewidth=3)

for mem in range(len(var_M.member)):
    ax.plot(range(59,len(var_M.time)+59), var_M.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)

ax.plot(range(len(var_J.time)), var_J.mean(dim='member'), color='green', label='Jan', linewidth=3)

for mem in range(len(var_J.member)):
    ax.plot(range(len(var_J.time)), var_J.sel(member=mem), color='green',  linewidth=0.8, alpha=0.5)


ax.set_xlim(0,150)
plt.legend(fontsize=22)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Zonal mean zonal wind, 60°N, 10 hPa (m/s)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/U_evolution_min_restart_0008.pdf')
plt.savefig('/home/weiji/test code/plots/U_evolution_min_restart_0008.png')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(31,len(var_F.time)+31), var_F.mean(dim='member'), color='royalblue', label='Feb, small pertlim', linewidth=3)

for mem in range(len(var_F.member)):
    ax.plot(range(31,len(var_F.time)+31), var_F.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_F2.time)+31), var_F2.mean(dim='member'), color='teal', label='Feb, large pertlim', linewidth=3)

for mem in range(len(var_F2.member)):
    ax.plot(range(31,len(var_F2.time)+31), var_F2.sel(member=mem), color='teal',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_F3.time)+31), var_F3.mean(dim='member'), color='midnightblue', label='Feb, Q pert', linewidth=3)

for mem in range(len(var_F3.member)):
    ax.plot(range(31,len(var_F3.time)+31), var_F3.sel(member=mem), color='midnightblue',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_FNC.time)+31), var_FNC.mean(dim='member'), color='red', label='Feb, NOCOUPL', linewidth=3)

for mem in range(len(var_FNC.member)):
    ax.plot(range(31,len(var_FNC.time)+31), var_FNC.sel(member=mem), color='red',  linewidth=0.8, alpha=0.5)


ax.set_xlim(0,150)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)

plt.legend(fontsize=22)
    
ax.set_ylabel('Zonal mean zonal wind, 60°N, 10 hPa (m/s)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/U_evolution_min_restart_0008_Feb.pdf')
plt.savefig('/home/weiji/test code/plots/U_evolution_min_restart_0008_Feb.png')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(59,len(var_M.time)+59), var_M.mean(dim='member'), color='hotpink', label='Mar, small pertlim', linewidth=3)

for mem in range(len(var_M.member)):
    ax.plot(range(59,len(var_M.time)+59), var_M.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)

ax.plot(range(59,len(var_M2.time)+59), var_M2.mean(dim='member'), color='purple', label='Mar, large pertlim', linewidth=3)

for mem in range(len(var_M2.member)):
    ax.plot(range(59,len(var_M2.time)+59), var_M2.sel(member=mem), color='purple',  linewidth=0.8, alpha=0.5)

    
ax.plot(range(59,len(var_M3.time)+59), var_M3.mean(dim='member'), color='blueviolet', label='Mar, Q pert', linewidth=3)

for mem in range(len(var_M3.member)):
    ax.plot(range(59,len(var_M3.time)+59), var_M3.sel(member=mem), color='blueviolet',  linewidth=0.8, alpha=0.5)
    
plt.legend(fontsize=22)
    
ax.set_xlim(0,150)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Zonal mean zonal wind, 60°N, 10 hPa (m/s)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/U_evolution_min_restart_0008_Mar.pdf')
plt.savefig('/home/weiji/test code/plots/U_evolution_min_restart_0008_Mar.png')

## Temperature plots for different intialisations

In [ ]:
#WACCM INT_O3

nc_fid=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc')

In [ ]:
O3=nc_fid['T']

O3=O3.sel(plev=5000)

O3=O3.sel(time=nc_fid.time.dt.month.isin([1,2,3,4,5,6]))
O3=O3.mean(dim='lon')

var=O3.sel(lat=slice(60,90)).min(dim='lat')

var_0008=var.sel(time=var.time.dt.year.isin([8]))

In [ ]:
var_0008_clim=var.groupby("time.dayofyear").mean()

In [ ]:
nc_fid_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F=nc_fid_F['T']
o3_F=o3_F.mean(dim='lon')

o3_F=o3_F.sel(plev=5000)

var_F=o3_F.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_F2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F2=nc_fid_F2['T']
o3_F2=o3_F2.mean(dim='lon')

o3_F2=o3_F2.sel(plev=5000)

var_F2=o3_F2.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_F3=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F3=nc_fid_F3['T']
o3_F3=o3_F3.mean(dim='lon')

o3_F3=o3_F3.sel(plev=5000)

var_F3=o3_F3.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_FNC=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_FNC=nc_fid_FNC['T']
o3_FNC=o3_FNC.mean(dim='lon')

o3_FNC=o3_FNC.sel(plev=5000)

var_FNC=o3_FNC.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_M=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M=nc_fid_M['T']
o3_M=o3_M.mean(dim='lon')

o3_M=o3_M.sel(plev=5000)

var_M=o3_M.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_M2=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M2=nc_fid_M2['T']
o3_M2=o3_M2.mean(dim='lon')

o3_M2=o3_M2.sel(plev=5000)

var_M2=o3_M2.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_M3=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc' ,concat_dim='member', combine='nested')

o3_M3=nc_fid_M3['T']
o3_M3=o3_M3.mean(dim='lon')

o3_M3=o3_M3.sel(plev=5000)

var_M3=o3_M3.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
nc_fid_J=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc' ,concat_dim='member', combine='nested')

o3_J=nc_fid_J['T']
o3_J=o3_J.mean(dim='lon')

o3_J=o3_J.sel(plev=5000)

var_J=o3_J.sel(lat=slice(60,90)).min(dim='lat')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

# plt.errorbar(range(0,92), var_highest_mean_before[len(var_lowest_mean)-92:len(var_lowest_mean)] ,var_highest_std_before[len(var_lowest_mean)-92:len(var_lowest_mean)], color='red', alpha=0.5,elinewidth=1.5, linewidth=3)
ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)
ax.plot(range(len(var_0008_clim)), var_0008_clim , color='grey', linestyle=':', linewidth=3)


ax.plot(range(31,len(var_F.time)+31), var_F.mean(dim='member'), color='royalblue', label='Feb', linewidth=3)

for mem in range(len(var_F.member)):
    ax.plot(range(31,len(var_F.time)+31), var_F.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)


ax.plot(range(59,len(var_M.time)+59), var_M.mean(dim='member'), color='hotpink', label='Mar',  linewidth=3)

for mem in range(len(var_M.member)):
    ax.plot(range(59,len(var_M.time)+59), var_M.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)

ax.plot(range(len(var_J.time)), var_J.mean(dim='member'), color='green', label='Jan', linewidth=3)

for mem in range(len(var_J.member)):
    ax.plot(range(len(var_J.time)), var_J.sel(member=mem), color='green',  linewidth=0.8, alpha=0.5)

ax.axhline(y=197, color='royalblue', linestyle='--')
ax.text(5,194, 'Cl activation threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

    
ax.set_xlim(0,150)
plt.legend(fontsize=22)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)
    
ax.set_ylabel('Minimum Temperature, 60-90°N, 50 hPa (K)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/T_min_evolution_min_restart_0008.pdf')
plt.savefig('/home/weiji/test code/plots/T_min_evolution_min_restart_0008.png')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(31,len(var_F.time)+31), var_F.mean(dim='member'), color='royalblue', label='Feb, small pertlim', linewidth=3)

for mem in range(len(var_F.member)):
    ax.plot(range(31,len(var_F.time)+31), var_F.sel(member=mem), color='royalblue',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_F2.time)+31), var_F2.mean(dim='member'), color='teal', label='Feb, large pertlim', linewidth=3)

for mem in range(len(var_F2.member)):
    ax.plot(range(31,len(var_F2.time)+31), var_F2.sel(member=mem), color='teal',  linewidth=0.8, alpha=0.5)

ax.plot(range(31,len(var_F3.time)+31), var_F3.mean(dim='member'), color='midnightblue', label='Feb, Q pert', linewidth=3)

for mem in range(len(var_F3.member)):
    ax.plot(range(31,len(var_F3.time)+31), var_F3.sel(member=mem), color='midnightblue',  linewidth=0.8, alpha=0.5)

   
ax.plot(range(31,len(var_FNC.time)+31), var_FNC.mean(dim='member'), color='red', label='Feb, NOCOUPL', linewidth=3)

for mem in range(len(var_FNC.member)):
    ax.plot(range(31,len(var_FNC.time)+31), var_FNC.sel(member=mem), color='red',  linewidth=0.8, alpha=0.5)


ax.axhline(y=197, color='royalblue', linestyle='--')
ax.text(5,194, 'Cl activation threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

    
ax.set_xlim(0,150)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)

plt.legend(fontsize=22)
    
ax.set_ylabel('Minimum Temperature, 60-90°N, 50 hPa (K)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/T_min_evolution_min_restart_0008_Feb.pdf')
plt.savefig('/home/weiji/test code/plots/T_min_evolution_min_restart_0008_Feb.png')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
sns.set_style("ticks")

ax.plot(range(len(var_0008)), var_0008 , color='grey',  linewidth=3)

ax.plot(range(59,len(var_M.time)+59), var_M.mean(dim='member'), color='hotpink', label='Mar, small pertlim', linewidth=3)

for mem in range(len(var_M.member)):
    ax.plot(range(59,len(var_M.time)+59), var_M.sel(member=mem), color='hotpink',  linewidth=0.8, alpha=0.5)

ax.plot(range(59,len(var_M2.time)+59), var_M2.mean(dim='member'), color='purple', label='Mar, large pertlim', linewidth=3)

for mem in range(len(var_M2.member)):
    ax.plot(range(59,len(var_M2.time)+59), var_M2.sel(member=mem), color='purple',  linewidth=0.8, alpha=0.5)

    
ax.plot(range(59,len(var_M3.time)+59), var_M3.mean(dim='member'), color='blueviolet', label='Mar, Q pert', linewidth=3)

for mem in range(len(var_M3.member)):
    ax.plot(range(59,len(var_M3.time)+59), var_M3.sel(member=mem), color='blueviolet',  linewidth=0.8, alpha=0.5)
    
plt.legend(fontsize=22)
    
ax.set_xlim(0,150)

ax.set_xticks([0,31,59,89,120])
ax.set_xticklabels( ['Jan','Feb','Mar','Apr','May', 'June'], fontsize=22)

ax.axhline(y=197, color='royalblue', linestyle='--')
ax.text(5,194, 'Cl activation threshold'  , horizontalalignment='left',fontsize=20,color='royalblue')

    
ax.set_ylabel('Minimum Temperature, 60-90°N, 50 hPa (K)', fontsize=22)
ax.set_yticklabels(ax.get_yticks(), size = 22)

plt.savefig('/home/weiji/test code/plots/T_min_evolution_min_restart_0008_Mar.pdf')
plt.savefig('/home/weiji/test code/plots/T_min_evolution_min_restart_0008_Mar.png')